## **Chemical Equilibrium - 2 - A Statistical-Mechanical Perspective**

<div align="left">
  <table border="1" cellpadding="6" cellspacing="0">
    <tr>
      <td bgcolor="#444444">
        <font color="#ffeb3b"><tt><b>Last updated (YYYY-MM-DD): 2026-01-08</b></tt></font>
      </td>
    </tr>
  </table>
</div>


### **Preparing the Computational Environment**  

*The next two cells prepare the environment by installing the necessary packages, importing libraries, and loading constants and functions. Make sure to run both before starting part (a) or part (b) of this notebook.*

⏳ **Note:** This may take **a few minutes** to complete. ⏳  

In [ ]:
#@title <small><small> { display-mode: "form" }
%%capture

# --- system (for LaTeX text in matplotlib) --- #
! sudo apt update -y
! sudo apt install -y cm-super dvipng texlive-latex-extra texlive-latex-recommended

# --- Python: install and update core scientific libraries --- #

# versions that are known to work well together will be selected to avoid compatibility issues
%pip install "numpy==2.0.2"
%pip install "scipy==1.16.3"

# install pyscf: as pyscf[geomopt,dispersion] gives problems, they are installed separately
%pip install --prefer-binary "pyscf==2.11.0"
%pip install "geometric==1.1"
%pip install "pyscf-dispersion==1.3.0"

%pip install "py3Dmol==2.5.3"
%pip install "pubchempy==1.0.5"
%pip install "rdkit==2025.9.1"

#!pip index versions pyscf # check available versions
# %pip install "pyberny==0.6.3"
# !pip show pyberny # check installed version

In [ ]:
#@title <small><small> { display-mode: "form" }

# ====    Import libraries of interest    ==== #
# -------------------------------------------- #
import os                                      #
# -------------------------------------------- #
import numpy                as np              #
#print(np.__version__)                         #
# -------------------------------------------- #
import matplotlib.pyplot    as plt             #
# -------------------------------------------- #
from   scipy.optimize  import minimize_scalar  #
from   scipy.optimize  import minimize         #
# -------------------------------------------- #
import pubchempy       as     pcp              # to retrieve from PubChem
# -------------------------------------------- #
import rdkit                                   # SMILES --> coordinates
from   rdkit           import RDLogger         #
from   rdkit           import Chem             #
from   rdkit.Chem      import AllChem          #
# -------------------------------------------- #
import py3Dmol                                 # 3D visualization of molecules
# -------------------------------------------- #
import pyscf                                   # for elec struct calculations
from   pyscf.geomopt   import geometric_solver #
from   pyscf.hessian   import thermo           #
# -------------------------------------------- #
import ipywidgets      as     w                # to add buttons
# -------------------------------------------- #
from   google.colab    import files            # to access to generated files
from   google.colab    import output           #
from   IPython.utils   import io               # to capture output
from   IPython.display import HTML             # needed for 3D visualization
from   IPython.display import display          # needed for 3D visualization
from   IPython.display import Markdown         # needed for 3D visualization
# ============================================ #


# ---- Define some constants of interest (SI) ----
m_u    = 1.66053886e-27      # atomic mass constant in kg
m_e    = 9.1093837E-31       # mass of electron
q_e    = 1.60217663E-19      # charge of electron
h      = 6.6260693E-34       # Planck's constant in J*s
k_B    = 1.3806505E-23       # Boltzmann's constant in J/K
c_0    = 2.99792558E8        # Speed of light in m/s
eps0   = 8.85418782E-12      # Vacuum permittivity
NA     = 6.02214076E23       # Avogadro's number
# ------------------------------------------------
P_o    = 1E5                 # standard pressure (P^o = 1E5 Pa = 1 bar)
R      = NA * k_B            # gas constant J/(K mol)
# ---- Universal constants ----> Atomic Units ----
hbar    = h/(2*np.pi)
a_0     = (4*np.pi*eps0*hbar**2) / (m_e * q_e**2)
Eh      = q_e**2 / (4*np.pi * eps0 * a_0)
Hz_au   = Eh/hbar
# ------------------------------------------------
ZERO1  = 1E-100               # |x|<ZERO --> x = ZERO
ZERO2  = 1E-010
ZERO3  = 1E-003
# ------------------------------------------------
last_fig = None
# ------------------------------------------------

# ---- Get reaction from the string given by user ----
def string_to_reaction(string):
    reactants, products = string.split("->")
    reactants = [i.strip() for i in reactants.split("+")]
    products  = [i.strip() for i in  products.split("+")]
    nus,molecules = [],[]
    for r in reactants:
        r = [i.strip() for i in r.split()]
        if   len(r) == 1: nu,molecule = 1,r[0]
        elif len(r) == 2: nu,molecule = r
        else            : raise Exception
        nus.append(-int(nu))
        molecules.append(molecule)
    for r in products:
        r = [i.strip() for i in r.split()]
        if   len(r) == 1: nu,molecule = 1,r[0]
        elif len(r) == 2: nu,molecule = r
        else            : raise Exception
        nus.append(+int(nu))
        molecules.append(molecule)
    return np.array(nus),np.array(molecules)

# ---- Write the equation of the reaction ----
def reaction_to_string(nus,molecules):
    stringR = []
    stringP = []
    for nu, molecule in zip(nus,molecules):
        if nu < 0: stringR += ["%s * %s"%(-nu,molecule)]
        if nu > 0: stringP += ["%s * %s"%(+nu,molecule)]
    return " + ".join(stringR) + " ⇌ " + " + ".join(stringP)

def level_to_string(functional,basis):
    level      = rf"{functional.upper():s}_{basis.upper():s}"
    # replace * by _ast_ to avoid name problems
    level      = level.replace("**","_ast_ast_")
    level      = level.replace("*","_ast_")
    return level

# ---- names of files of interets ----
def files_of_interest(molecule,functional="",basis=""):

    level      = level_to_string(functional,basis)
    # filenames
    xyz_guess  = rf"xyz_guess-{molecule:s}.xyz"
    xyz_opt    = rf"xyz_optim-{molecule:s}.{level:s}.xyz"
    output_opt = rf"pyscf_opt-{molecule:s}.{level:s}.out"
    output_frq = rf"pyscf_frq-{molecule:s}.{level:s}.out"
    # return filenames
    return xyz_guess,xyz_opt,output_opt,output_frq

# ---- PUBCHEM: get geom ----
def pubchem_cid(cid):
    symbols,coords,smiles = None,None,None

    try   : compound = pcp.Compound.from_cid(int(cid))
    except: compound = None

    if compound is not None:
       print(rf"     - geometry retrieved!")
       print(rf"")
       print(rf"     - information:")
       print(rf"       * PubChem CID       : {compound.cid}")
       print(rf"       * molecular formula : {compound.molecular_formula:s}")
       # print(rf"       * charge            : {compound.charge:d}")
       print(rf"       * SMILES            : '{compound.smiles:s}'")
       print("")
       smiles  = compound.smiles
       # for diatomic molecules, 3D geoemtry is not stored in PubChem
       try   : geom = pcp.get_compounds(int(cid),"cid",record_type='3d')[0]
       except: geom = pcp.get_compounds(int(cid),"cid")[0]
       symbols = [atom.element           for atom in geom.atoms]
       # z-coordinate may be None (when record_type='3d' is not used)
       coords  = [tuple(0.0 if c is None else c for c in (atom.x, atom.y, atom.z)) for atom in geom.atoms]
    return symbols,coords,smiles

# ---- RDKIT: get geom (from SMILES) ----
def rdkit_smiles2geom(smiles):

    symbols,coords = None,None
    try:
       m   = Chem.AddHs(Chem.MolFromSmiles(smiles))
       cid = AllChem.EmbedMolecule(m)
       if cid >= 0:
          symbols = [atom.GetSymbol() for atom in m.GetAtoms()]
          coords  = [list(m.GetConformer().GetAtomPosition(i)) for i in range(m.GetNumAtoms())]
          mformu  = {s:0 for s in symbols}
          for s in symbols: mformu[s] += 1
          mformu  = "".join([k+str(v) if v!=1 else k for k,v in mformu.items()])
          print(rf"     - geometry generated!")
          print(rf"")
          print(rf"     - information:")
          print(rf"       * molecular formula = {mformu:s}")
          print(rf"       * SMILES            = '{smiles:s}'")
          print("")
    except: pass

    return symbols,coords,smiles

# ---- button to download file ----
def download_file(fname):
    btn = w.Button(
            description=f"Download {fname}",
            icon="download",
            button_style="primary",
            layout=w.Layout(width='250px', height='38px', margin='0 0 0 55px')
    )
    # action when clicking
    btn.on_click(lambda _: files.download(fname))
    return btn

def _on_download_clicked(_,fname):
    # --- get figure from global variable ---
    fig = last_fig
    if fig is None:
       print("No figure yet; move a slider to generate the plot...")
       return
    fig.savefig(fname, bbox_inches='tight')
    files.download(fname)

# ---- data from smiles to .xyz file ----
def data_2_xyz(symbols,xcc,fname,smiles=""):
    string  = rf"{len(symbols):.0f}"+"\n"
    string += rf"Cartesian coordinates for SMILES: {smiles:s}"+"\n"
    for idx,symbol in enumerate(symbols):
        xx,yy,zz = [coord for coord in xcc[idx]]
        string += rf"{symbol:2s}     {xx:9.5f}  {yy:9.5f}  {zz:9.5f}" + "\n"
    with open(fname,'w') as asdf: asdf.write(string)
    # download file button
    btn = download_file(fname)
    display(btn)
    print("")

# ---- read .xyz file ----
def read_xyz(filename):
    with open(filename) as f: lines = f.readlines()
    nat = int(lines[0])
    symbols = []
    coords = []
    for line in lines[2:2+nat]:
        parts = line.split()
        if len(parts) < 4: continue
        sym, x, y, z = parts[:4]
        symbols.append(sym)
        coords.append([float(x), float(y), float(z)])
    return symbols, np.array(coords)

# ---- to visualize geom in .xyz file ----
def create_visualization_xyz(xyz_file):
    # draw molecule
    view = py3Dmol.view(width=300, height=200)
    view.addModel(open(xyz_file, 'r').read(), 'xyz')
    view.setStyle({'stick': {'singleBonds': False}, 'sphere': {'scale': 0.3}})
    view.zoomTo()
    view.zoom(2.5)
    return view

def geometric_info_xyz(xyz_file,geominfo):
    info = ""
    symbols, coords = read_xyz(xyz_file)
    for ii in geominfo:
        if len(ii) == 2:
          at1,at2 = ii
          distance = np.linalg.norm(coords[at1] - coords[at2])
          sbond    = rf"{symbols[at1]:s}-{symbols[at2]:s}"
          info    += rf"       * d({sbond:5s}) = {distance:.3f} Å" + "\n"
        elif len(ii) == 3:
          at1,at2,at3 = ii
          v1 = coords[at1] - coords[at2]
          v2 = coords[at3] - coords[at2]
          cosang = np.dot(v1, v2) / (np.linalg.norm(v1)*np.linalg.norm(v2))
          cosang = np.clip(cosang, -1.0, 1.0)
          angle  = np.degrees(np.arccos(cosang))
          sangle = rf"{symbols[at1]:s}-{symbols[at2]:s}-{symbols[at3]:s}"
          info  += rf"       * ∠({sangle:5s}) = {angle:.1f}°" + "\n"
    return info

# ---- geometry optization with PySCF ----
def pyscf_carryout_opt(molecule,unpaired,charge,functional,basis,bsym=False):

    # Files of interest
    xyz_guess,xyz_opt,output_opt,output_frq = files_of_interest(molecule,functional,basis)

    # Generate molecule from .xyz file
    with io.capture_output() as captured:
        mol = pyscf.gto.Mole()
        mol.atom         = xyz_guess
        mol.spin         = unpaired
        mol.charge       = charge
        mol.basis        = basis
        mol.output       = output_opt
        mol.verbose      = 4
        if bsym:
           mol.symmetry     = True
           mol.symmetrize   = True
           mol.symmetry_tol = 1e-2
        mol.build()

    # Define DFT method and mesh grid (from 0 to 9, default = 3)
    if unpaired == 0: mf = mol.RKS(xc=functional)
    else            : mf = mol.UKS(xc=functional)
    mf.grids.level = 5
    mf.max_cycle   = 200
    mf.conv_tol    = 1e-7

    # run SCF and avoid printing information
    with io.capture_output() as captured: mf   = mf.run()
    Etot = mf.e_tot
    print(rf"     - Etot(guess geometry) = {Etot:.5f} hartree")

    # optimization [avoid printing information]
    print("     - geometry optimization...")
    conv_params = {}
    conv_params['convergence_energy'] = 5.0e-7  # Eh
    conv_params['convergence_gmax'  ] = 2.0e-4  # Eh/Bohr
    with io.capture_output() as captured:
       try:
          opt_geom = geometric_solver.optimize(mf, maxsteps=300, **conv_params)
       except:
          conv_params['convergence_energy'] = 1.0e-6  # Eh
          conv_params['convergence_gmax'  ] = 1.0e-4  # Eh/Bohr
          opt_geom = geometric_solver.optimize(mf, maxsteps=300, **conv_params)

    pyscf.gto.tofile(opt_geom,xyz_opt)

# ---- frequency calculation with PySCF ----
def pyscf_carryout_frq(molecule,unpaired,charge,functional,basis,bsym=False):

    # Files of interest
    xyz_guess,xyz_opt,output_opt,output_frq = files_of_interest(molecule,functional,basis)

    # Generate molecule from optimized geometry stored in xyz file
    with io.capture_output() as captured:
        mol = pyscf.gto.Mole()
        mol.atom         = xyz_opt
        mol.spin         = unpaired
        mol.charge       = charge
        mol.basis        = basis
        mol.output       = output_frq
        mol.verbose      = 6
        if bsym:
           mol.symmetry     = True
           mol.symmetrize   = True
           mol.symmetry_tol = 1e-2
        mol.build()

    # Define DFT method and mesh grid
    if unpaired == 0: mf = mol.RKS(xc=functional)
    else            : mf = mol.UKS(xc=functional)
    mf.grids.level = 5
    mf.max_cycle   = 200
    mf.conv_tol    = 1e-7

    # run SCF and avoid printing information
    with io.capture_output() as captured: mf   = mf.run()
    Etot = mf.e_tot
    print(rf"     - Etot(optim geometry) = {Etot:.5f} hartree")

    # Carry out Hessian calculation
    print("     - Hessian calculation...")
    with io.capture_output() as captured: hessian = mf.Hessian().kernel()

    # add Hessian matrix to output_frq
    H4 = np.asarray(hessian)
    with open(output_frq, "a") as f:
        f.write("\n*** HESSIAN BY 3x3 ATOMIC BLOCKS (Hartree/Bohr^2) ***\n")
        for i in range(mol.natm):
            for j in range(mol.natm):
                f.write(f"\n# Block ({i+1},{j+1})  [atom {i+1} vs atom {j+1}]\n")
                np.savetxt(f, H4[i, j], fmt=" % .6e")

    # return data
    return mol, mf, hessian

# ---- download PySCF files ----
def pyscf_download(molecule,functional,basis,which_ones=[]):

    xyz_guess,xyz_opt,output_opt,output_frq = files_of_interest(molecule,functional,basis)
    print(rf"     - file(s) to download:")
    if 1 in which_ones:
         print(rf"       {xyz_opt:s}")
         btn1 = download_file(xyz_opt)   ; display(btn1)
    if 2 in which_ones:
         print(rf"       {output_opt:s}")
         btn2 = download_file(output_opt); display(btn2)
    if 3 in which_ones:
         print(rf"       {output_frq:s}")
         btn3 = download_file(output_frq); display(btn3)

# ---- information of interest after Hessian calc ----
def pyscf_extract(mol, mf, hessian, unpaired):

    # translational info
    masses  = mol.atom_mass_list(isotope_avg=True)
    mass    = sum(masses)

    # vibrational info
    freqs       = thermo.harmonic_analysis(mf.mol, hessian)['freq_au']
    au2hz       = (1/2/np.pi)*(Eh/m_u/a_0**2)**0.5
    freqs_Hz    = [f*au2hz for f in freqs]
    wavenum_m   = [f/c_0   for f in freqs_Hz]

    info_thermo = thermo.thermo(mf, freqs) # by default, at 298.15 K and 101325 Pa
    ZPE         = info_thermo['ZPE'][0]

    # rotational info
    A,B,C = thermo.rotation_const(masses,mol.atom_coords(),unit='GHz')
    # if linear, two are equal, the other is infinity
    if   np.isinf(A) and abs(B-C) < ZERO2: A,B,C,linear = B,None,None,True
    elif np.isinf(B) and abs(A-C) < ZERO2: A,B,C,linear = C,None,None,True
    elif np.isinf(C) and abs(A-B) < ZERO2: A,B,C,linear = A,None,None,True
    else                                 :       linear =             False
    sigma = thermo.rotational_symmetry_number(mol)

    # electronic info
    E0 = info_thermo['E0' ][0]

    # collect data
    data = {}
    data["natoms"  ] = mol.natm
    data["mass"    ] = mass
    data["rotcons" ] = [i*1E9 if i is not None else i for i in [A,B,C]] # in Hz
    data["rotsigma"] = sigma
    data["islinear"] = linear
    data["freqs"   ] = wavenum_m # in 1/m
    data["ZPE"     ] = ZPE       # in hartree
    data["unpaired"] = unpaired
    data["E0"      ] = E0        # in hartree

    # return data
    return data

# ---- print information of interest after Hessian calc ----
def pyscf_printdata(data):

    # unpack data
    mass     = data["mass"    ]
    A,B,C    = data["rotcons" ]
    linear   = data["islinear"]
    sigma    = data["rotsigma"]
    freqs    = data["freqs"   ]
    ZPE      = data["ZPE"     ]
    unpaired = data["unpaired"]
    E0       = data["E0"      ]

    # print fata
    INFO  = rf"     - total mass (amu)              : {mass:.2f}" + "\n"
    if linear: INFO += rf"     - rotational constant  (GHz)    : {(A*1E-9):.2f}" + "\n"
    else     : INFO += rf"     - rotational constants (GHz)    : " + "  ".join(["%.2f"%(ii*1E-9) for ii in (A,B,C)]) + "\n"
    INFO += rf"     - rotational symmetry number    : {sigma:d}" + "\n"
    INFO += rf"     - vibrational wavenumbers (1/cm):"+"\n"
    # frequencies are, actually, wavenumbers in (1/m)
    for i in range(0,len(freqs),5):
        INFO += "         "+"  ".join(["%8.2f"%(ii/100) for ii in freqs[i:i+5]])+"\n"
    INFO += rf"     - zero point energy (hartree)   : {ZPE:.5f}"+"\n"
    print(INFO)

# ---- OPT + FREQ CALCULATION ----
def optimize_and_freqs(molecule,unpaired,charge,functional,basis,bsym=False):
    # files
    xyz_guess,xyz_opt,output_opt,output_frq = files_of_interest(molecule,functional,basis)
    # check if calculation was already carried out
    args = (molecule,unpaired,charge,functional,basis,bsym)
    # geometry optimization
    if not os.path.exists(xyz_opt): pyscf_carryout_opt(*args)
    # Hessian calculation
    mol, mf, hessian = pyscf_carryout_frq(*args)
    # Extract data
    return pyscf_extract(mol,mf,hessian,unpaired)

# ---- translational partition function ----
def pfn_translational(T,mass):

    beta    = 1/(k_B*T)

    # to SI
    mass_SI = mass * m_u

    # translational contribution
    V_per_molec = 1 / (beta * P_o)
    broglie_wvl = ((beta * h**2) / (2 * np.pi * mass_SI))**(0.5)
    q_translat  = V_per_molec / broglie_wvl**3

    # d(lnq_tr)/dbeta at constant volume
    dlnqdbeta_v = - 3/2 * (1/beta)

    # d(lnq_tr)/dbeta at constant pressure
    dlnqdbeta_p = - 5/2 * (1/beta)

    return q_translat, dlnqdbeta_v

# ---- rotational partition function ----
def pfn_rotational(T,A,B,C,linear,sigma):

    beta = 1/(k_B*T)

    # rotational constants (Hz --> 1/m) and rot temperature
    A /= c_0; theta_A = (h*c_0/k_B) * A

    if linear:
      q_rotational = T / theta_A
      dlnqdbeta    = - (1/beta)
    else:
      B /= c_0; theta_B = (h*c_0/k_B) * B
      C /= c_0; theta_C = (h*c_0/k_B) * C
      q_rotational = (np.pi * T**3 / (theta_A * theta_B * theta_C))**(1/2)
      dlnqdbeta    = - 3/2 * (1/beta)

    q_rotational /= sigma

    return q_rotational, dlnqdbeta

# ---- vibrational partition function ----
def pfn_vibrational(T,freqs):

    beta = 1/(k_B*T)

    q_vibrational = 1.0
    dlnqdbeta     = 0.0
    for freq in freqs:
        nu    = freq * c_0 # in Hz
        theta = h*nu/k_B
        q_vibrational *= 1 / (1 - np.exp(-theta/T)) # from ZPE
        dlnqdbeta     += - h*nu / (np.exp(theta/T) - 1)

    return q_vibrational, dlnqdbeta

# ---- electronic partition function ----
def pfn_electronic(T,unpaired):

    q_electronic = unpaired + 1
    dlnqdbeta    = 0.0

    return q_electronic, dlnqdbeta

# ---- compute U, H, S, and G ----
def compute_thermodynamics(T,molecule,key,thermodata):

    # unpack data
    natoms   = thermodata[molecule][key]["natoms"  ]
    mass     = thermodata[molecule][key]["mass"    ]
    A,B,C    = thermodata[molecule][key]["rotcons" ]
    linear   = thermodata[molecule][key]["islinear"]
    sigma    = thermodata[molecule][key]["rotsigma"]
    freqs    = thermodata[molecule][key]["freqs"   ]
    unpaired = thermodata[molecule][key]["unpaired"]
    E0       = thermodata[molecule][key]["E0"      ]
    ZPE      = thermodata[molecule][key]["ZPE"     ]

    # Get partition functions
    q_translat   , dlnqtdbeta = pfn_translational(T,mass)
    if natoms == 1:
       q_rotational , dlnqrdbeta = 1.0, 0.0
       q_vibrational, dlnqvdbeta = 1.0, 0.0
    else:
       q_rotational , dlnqrdbeta = pfn_rotational(T,A,B,C,linear,sigma)
       q_vibrational, dlnqvdbeta = pfn_vibrational(T,freqs)
    q_electronic , dlnqedbeta = pfn_electronic(T,unpaired)
    q_tot         = q_translat * q_rotational * q_vibrational * q_electronic
    dlnqdbeta_tot = dlnqtdbeta + dlnqrdbeta + dlnqvdbeta + dlnqedbeta

    # Info line
    line  = rf" {q_translat:.4E} | {q_rotational:.4E} | {q_vibrational:.4E} | {q_electronic:.4E} | {q_tot:.4E} | {(E0+ZPE):12.5f}"

    # calculate U and S (in S.I.; per molecule)
    Eref = (E0+ZPE)*Eh
    U    = Eref - dlnqdbeta_tot
    S    = - dlnqdbeta_tot/T + k_B * np.log(q_tot * np.e)

    # calculate H and G (per molecule)
    H    = U + k_B * T
    G    = H - S   * T

    # return data in J and J/K
    return U, H, S, G, line

# ---- Delta{G}^o as a function of T ----
def get_DGo(T,T_ref,DHo_ref,DSo_ref,DCPo_ref):
    DGo_T  = DHo_ref - T * DSo_ref
    DGo_T += DCPo_ref  * ( T - T_ref + T*np.log(T_ref / T))
    return DGo_T

# ---- Functions for fitting: best valur of Delta_{r}{C}_p^o ----
def sum_squared_errors(DCP,T,DGo_T,T_ref,DHo_ref,DSo_ref):
    DCPo_ref  = DCP * R
    DGo_model = get_DGo(T,T_ref,DHo_ref,DSo_ref,DCPo_ref)
    residuals = DGo_model - DGo_T
    return np.sum(residuals**2)

def calculate_rms(x1,x2):
    return (sum([(i-j)**2 for i,j in zip(x1,x2)])/len(x1))**0.5

# ---- plot DGo with temperature ----
def plot_DG_T(T,DGo_T,DGo_model,DCP,T_ref,DGo_ref,key):

    plt.rcParams['text.usetex'] = True

    if DCP is not None: DCP = DCP/R
    # Plot results
    plt.plot(T,[ii/1000 for ii in DGo_T]    ,'k-',zorder=1,label=r"Stat-Mech")

    if DGo_model is not None:
       plt.plot(T,[ii/1000 for ii in DGo_model],'r--',zorder=1,label=r"Eq. (1) with $\Delta_{r}C_{p}^\circ= %.2f\cdot R$"%DCP)

    if T_ref is not None:
       plt.plot(T_ref,DGo_ref/1000,'ro')

    # Format plot
    plt.xticks(fontsize=14)
    plt.yticks(fontsize=14)

    plt.xlabel(r'$T \;\; (\mathrm{K})$'                        ,fontsize=16)
    plt.ylabel(r'$\Delta_{r} G^{\circ} \;\; (\mathrm{kJ/mol})$',fontsize=16)

    plt.legend(loc="best",fontsize=12)

    if DGo_model is not None:
       level = level_to_string(key[0],key[1])
       fname = rf"plot_DGo_T_stat_{level:s}_{DCP:5.2f}.svg"
       # ---- download button ----
       fig = plt.gcf()
       fig.savefig(fname, bbox_inches='tight')
       btn = download_file(fname)
       display(btn)

    # ---- show plot and close ----
    plt.show()
    plt.close()

#=====================================================================#
def freq_to_nu_and_theta(freq_cm):
    nu    = 100*freq_cm*c_0
    theta = h*nu/k_B
    return nu, theta
#---------------------------------------------------------------------#
def vib_contribution(freq_cm,T):
    nu,theta = freq_to_nu_and_theta(freq_cm)
    thetaT   = theta/T
    contri   = (thetaT * np.exp(-thetaT/2)/(1-np.exp(-thetaT))) **2
    return contri
#---------------------------------------------------------------------#
def vib_contri_avera(freq_cm,T1,T2):
    nu,theta = freq_to_nu_and_theta(freq_cm)
    average  = 1/(np.exp(theta/T2)-1) - 1/(np.exp(theta/T1)-1)
    average  = average * theta/(T2-T1)
    return average
#---------------------------------------------------------------------#
def plot3D_vibdof():
    # Frequencies (1/cm)
    xx    = np.linspace(1 ,2000,201)
    # Temperatures (K)
    yy    = np.linspace(10, 500,201) # temperature
    # Contribution (0,1)
    xx,yy = np.meshgrid(xx,yy)
    zz    = vib_contribution(xx,yy)

    # Generate plot
    fig = go.Figure()
    fig.add_trace(go.Surface(x=xx,y=yy,z=zz,colorscale='Viridis',showscale=True))
    fig.update_layout(
        width=650, height=500,
        margin=dict(l=0, r=0, t=40, b=0),
        scene=dict(
            xaxis=dict(title=dict(text='freq (1/cm)' ,font=dict(size=18))),
            yaxis=dict(title=dict(text='T (K)'       ,font=dict(size=18))),
            zaxis=dict(title=dict(text='contribution',font=dict(size=18))),
            camera=dict(eye=dict(x=1.6, y=1.6, z=0.9))
        ))

    options = {"format":"svg","filename":"equilibrium_surface","scale":2}
    config  = {"toImageButtonOptions":options}
    fig.show(config=config)
#---------------------------------------------------------------------#
def plot_average(Tmin,Tmax,freqmin,freqmax,freq=None):
    freqs    = np.linspace(freqmin,freqmax,101)
    averages = vib_contri_avera(freqs,Tmin,Tmax)
    yy_inf   = vib_contribution(freqs,Tmin)
    yy_sup   = vib_contribution(freqs,Tmax)
    plt.plot(freqs,yy_inf  ,'r--',label=rf"$T$={Tmin:.2f} K",zorder=2)
    plt.plot(freqs,yy_sup  ,'b--',label=rf"$T$={Tmax:.2f} K")
    plt.plot(freqs,averages,'k-' ,label=rf"average")
    plt.xlabel(r"Vib. frequency (cm$^{-1}$)"     ,fontsize=14)
    plt.ylabel(r"Vib. contribution, $n^{V^\ast}$",fontsize=14)
    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)

    # --- update global variable: last_fig ---
    global last_fig
    last_fig = plt.gcf()

    if freq is not None:
       # limits so far
       xlim = plt.gca().get_xlim()
       ylim = plt.gca().get_ylim()
       # get average contribution for selected frequency
       aver =  vib_contri_avera(freq,Tmin,Tmax)
       # plot data for selected frequency
       plt.plot([freq,freq],[ylim[0],aver],'--',color="grey",zorder=1)
       plt.plot([xlim[0],freq],[aver,aver],'--',color="grey",zorder=1)
       plt.plot(freq,aver,'o',color="grey",label=rf"$n^{{V^\ast}} = {aver:.3f}$")
       # keep origonal limits
       plt.xlim(xlim)
       plt.ylim(ylim)
    plt.legend(loc="best",fontsize=11)

    # --- Show and close figure ---
    plt.show()
    plt.close()
#=====================================================================#

### **(a) The worked example**: $\rm{N_2O_4(g)}\ \rightleftharpoons\ 2 NO_2(g)$ reaction


####
Let us revisit the dissociation reaction of $\rm N_2O_4 (g)$:

$$\rm N_2O_4 (g) \rightleftharpoons 2 \; NO_2 (g)$$

Run the following cell to load the reaction.



In [ ]:
#@title <small><small> { display-mode: "form" }

#------------------------------------------------------
MOLECULES     = ["N2O4","NO2"]
NUS           = np.array([-1,2])
MOLECULES_ID  = {k:None for k in MOLECULES}
THERMODATA    = {k:{}   for k in MOLECULES}
#------------------------------------------------------
GEOMINFO      = {"N2O4":[(1,5),(4,5),(1,5,3)] , "NO2":[(0,1),(1,0,2)]}
#------------------------------------------------------
string = reaction_to_string(NUS,MOLECULES)
print(rf"The reaction is: {string:s}")

#------------------------------------------------------
FUNCTIONALS   = "b3lyp-d4,o3lyp-d4,pbe0-d4".split(",")
BASIS         = "6-31g*"
#------------------------------------------------------

#### _(a.1) DFT calculations_


#####
We first need an initial guess for the geometries of the reactant (N$_2$O$_4$) and the product (NO$_2$).

<br>

We will retrieve the geometry for N$_2$O$_4$ from PubChem[[🌐]](https://pubchem.ncbi.nlm.nih.gov/) by using its PubChem CID identifier[[🌐]](https://pubchem.ncbi.nlm.nih.gov/compound/25352):

* PubChem CID for N$_2$O$_4$ is "25352"

Run the cell below and enter this identifier. Do _not_ include quotation marks when entering the CID.
Once the geometry is retrieved, it will be displayed in an interactive viewer that allows you to rotate and inspect the molecule freely.

In [ ]:
#@title <small><small> { display-mode: "form" }

molecule = "N2O4"
print("Fetching structures from PubChem 🔎 .\n")
print("Please enter the CID to search for:")
MOLECULES_ID[molecule] = input(rf"     * reactant ({molecule:s}): ").strip()
print("")

mol_id    = MOLECULES_ID[molecule]
xyz_guess = files_of_interest(molecule)[0]

# Get geometry
print(rf"Retrieving geometry for: {str(mol_id):s}")
print("")
symbols,coords,smiles = pubchem_cid(mol_id)

# Generate file
if symbols is None:
   print(rf"     - ERROR: unable to get geometry for: '{str(mol_id):s}'")
   print("")
else:
   data_2_xyz(symbols,coords,xyz_guess,smiles)
   print(rf"     - geometry stored in '{xyz_guess:s}'")
   info = geometric_info_xyz(xyz_guess,GEOMINFO[molecule])
   print(info)
   print("")
   view = create_visualization_xyz(xyz_guess)
   view.show()

#####

For NO$_2$ we will use an alternative way to obtain a reference structure: building the molecule from its SMILES representation[[🌐]](https://en.wikipedia.org/wiki/Simplified_Molecular_Input_Line_Entry_System) using with the **Rdkit** toolkit[[🌐]](https://www.rdkit.org/). You can find the SMILES for nitrogen dioxide online (including on Wikipedia[[🌐]](https://en.wikipedia.org/wiki/Nitrogen_dioxide)):

*  SMILES for NO$_2$: "[N+]\(=O)[O-]":

Run the following cell and paste the SMILES string to reconstruct a **3D geometry** from the code.

In [ ]:
#@title <small><small> { display-mode: "form" }

molecule = "NO2"

# mute warnings/erros in console
RDLogger.logger().setLevel(RDLogger.CRITICAL)

MOLECULES_ID[molecule] = input(rf"insert the SMILES code for {molecule:s} : ").strip()
if MOLECULES_ID[molecule].startswith('"'): MOLECULES_ID[molecule] = MOLECULES_ID[molecule][1:]
if MOLECULES_ID[molecule].endswith('"')  : MOLECULES_ID[molecule] = MOLECULES_ID[molecule][:-1]
print("")

# Get geometry
try:
   mol_id    = MOLECULES_ID[molecule]
   xyz_guess = files_of_interest(molecule)[0]
   print(rf"Retrieving geometry for: {str(mol_id):s}")
   print("")
   symbols,coords,smiles = rdkit_smiles2geom(mol_id)
   data_2_xyz(symbols,coords,xyz_guess,smiles)
   print(rf"     - geometry stored in '{xyz_guess:s}'")
   info = geometric_info_xyz(xyz_guess,GEOMINFO[molecule])
   print(info)
   print("")
   view = create_visualization_xyz(xyz_guess)
   view.show()

except:
   print(rf"ERROR! Something went wrong! Did you introduce the correct SMILES??")
   print("")

#####
The molecular structures obtained so far are **not** optimized; they do not correspond to the **minimum-energy geometries**.

Before performing any optimization, however, we must specify for each molecule both its total charge and the number of unpaired electrons. In our case:

* N$_2$O$_4$: **neutral** molecule with **no** unpaired electrons
* NO$_2$ &nbsp;: **neutral** molecule with **one** unpaired electron

Run the next cell and enter the value for these variables:

In [ ]:
#@title <small><small> { display-mode: "form" }

CHARGES,UNPAIREDS = {},{}
print("Insert the total charge (a.u.) and number of unpaired electrons for each molecule (integers only):")
print("")
for molecule in MOLECULES:
    print(rf"   * {molecule:s}")
    charge   = input("     total molecular charge [in au] = ").strip()
    unpaired = input("     number of unpaired electrons   = ").strip()
    CHARGES[molecule]   = int(charge)
    UNPAIREDS[molecule] = int(unpaired)
    print("")

#####

With this information in hand, we are ready to perform the quantum-chemistry calculations using **PySCF**[[🌐]](https://pyscf.org). After optimizing the geometries, we will also compute the **Hessian matrix** to verify that each structure corresponds to a true minimum (all vibrational frequencies are real), rather than a different type of stationary point. Moreover, these vibrational frequencies are required to obtain the vibrational partition function, which is essential for evaluating thermodynamic quantities.

We will test the following DFT levels of theory:

* B3LYP-D4
* O3LYP-D4
* PBE0-D4

with the **6-31G*** basis set.

**Note 1.** In PySCF, the selected basis set corresponds to "**6-31G*** **5d**" in Gaussian electronic structure software [[🌐]](https://gaussian.com/basissets/).

**Note 2.** ⏳ The execution of each cell may take **a few minutes** to complete (~10-15 min each cell). ⏳

In [ ]:
#@title <small>💻 Optimizing guess structures with B3LYP-D4 <small> { display-mode: "form" }

key = (FUNCTIONALS[0],BASIS)
print(rf" - Functional: {key[0].upper():s}")
print(rf" - Basis set : {key[1].upper():s}")
print("")
for molecule in MOLECULES:
    print(rf" * Molecule: {molecule:s}")
    THERMODATA[molecule][key] = optimize_and_freqs(molecule,UNPAIREDS[molecule],CHARGES[molecule],key[0],key[1],bsym=True)
    pyscf_printdata(THERMODATA[molecule][key])
    pyscf_download(molecule,key[0],key[1],[1])
    print("")

In [ ]:
#@title <small>💻 Optimizing guess structures with O3LYP-D4 <small> { display-mode: "form" }

key = (FUNCTIONALS[1],BASIS)
print(rf" - Functional: {key[0].upper():s}")
print(rf" - Basis set : {key[1].upper():s}")
print("")
for molecule in MOLECULES:
    print(rf" * Molecule: {molecule:s}")
    THERMODATA[molecule][key] = optimize_and_freqs(molecule,UNPAIREDS[molecule],CHARGES[molecule],key[0],key[1],bsym=True)
    pyscf_printdata(THERMODATA[molecule][key])
    pyscf_download(molecule,key[0],key[1],[1])
    print("")

In [ ]:
#@title <small>💻 Optimizing guess structures with PBE0-D4 <small> { display-mode: "form" }

key = (FUNCTIONALS[2],BASIS)
print(rf" - Functional: {key[0].upper():s}")
print(rf" - Basis set : {key[1].upper():s}")
print("")
for molecule in MOLECULES:
    print(rf" * Molecule: {molecule:s}")
    THERMODATA[molecule][key] = optimize_and_freqs(molecule,UNPAIREDS[molecule],CHARGES[molecule],key[0],key[1],bsym=True)
    pyscf_printdata(THERMODATA[molecule][key])
    pyscf_download(molecule,key[0],key[1],[1])
    print("")

#####
Run the cell below to **visualize** the optimized geometries. Again, the views are interactive; you can rotate and inspect the molecule freely.

In [ ]:
#@title <small><small> { display-mode: "form" }

print("")
for molecule in MOLECULES:

    html_blocks = []
    print(rf"==> molecule: {molecule:s}")
    for functional in FUNCTIONALS:

        xyz_opt = files_of_interest(molecule,functional,BASIS)[1]
        if not os.path.exists(xyz_opt): continue
        # generate visualization of molecule
        view = create_visualization_xyz(xyz_opt)
        # read xyz file and get some interesting geometric features
        info  = functional+"/"+BASIS + "\n"
        info += geometric_info_xyz(xyz_opt,GEOMINFO[molecule])

        # save for later visualization
        html_blocks.append(f"""
            <div style='text-align:center; margin:10px;'>
                {view._make_html()}
                <div style='font-size:16px; font-weight:500; margin-top:-2px;white-space:pre-line;'>
                    {info}
                </div>
            </div>
        """)

    # Displays all viewers side by side
    html_code = "<div style='display:flex; gap:100px; width:100%;'>" + "".join(html_blocks) + "</div>"
    display(HTML(html_code))
    print("")

#####

PySCF does not always return the correct symmetry number ($\sigma$) due to numerical inaccuracies [[🌐]](https://link.springer.com/article/10.1007/s00214-007-0328-0). Therefore, before moving to (a.2) and compute the partition functions, we must ensure that the symmetry numbers are correct. For the species involved in the reaction studied here, the appropriate values are:

* N$_2$O$_4$ ==> D2h symmetry ==> $\sigma = 4$
* NO$_2$ &nbsp;  ==> C2v symmetry ==> $\sigma = 2$

Verify these values in the preceding optimization cells, and run the following cell if any symmetry numbers need to be corrected.

In [ ]:
#@title <small>💻 Correct symmetry numbers <small> { display-mode: "form" }
print("Insert the rotational symmetry number for each molecule:")
print("")
for molecule in MOLECULES:
    sigma = int(input(rf"   * sigma({molecule:s}) = ").strip())
    print("")
    for functional in FUNCTIONALS:
        key = (functional,BASIS)
        try:
          sigma_old = THERMODATA[molecule][key]["rotsigma"]
          print(rf"     {functional.upper():8s}: {sigma_old:d} --> {sigma:d}")
          THERMODATA[molecule][key]["rotsigma"] = sigma
        except:
          print(rf"     data not found for {functional.upper():s}...")
    print("")

#### _(a.2) Calculation of partition functions and thermodynamic functions_


#####
We can now evaluate the partition functions of all species participating in the reaction at 298.15 K. From them, the thermodynamic state functions can be determined.

In [ ]:
#@title <small><small> { display-mode: "form" }

T_ref = 298.15

TABLE  = " ----------------------------------------------------------------------------------------------" + "\n"
TABLE += "  FUNCTIONAL/BASIS_SET | DELTA_r{U}^o | DELTA_r{H}^o | DELTA_r{S}^o | DELTA_r{G}^o |   K_p^o   " + "\n"
TABLE += " ----------------------|--------------|--------------|--------------|--------------|-----------" + "\n"

print("Partition functions and zero-point–corrected total energies (E0 + ZPE); energies in hartree.")
print("")

REFDATA = {}
for functional in FUNCTIONALS:
    key   = (functional,BASIS)
    level = rf"{key[0].upper():8s}/{key[1].upper():s}"
    allok = True

    SUBTABLE  = "    --------------------------------------------------------------------------------------------"+"\n"
    SUBTABLE += "      molecule |   pfn tr    |  pfn rot   |  pfn vib   |  pfn ele   |  pfn tot   |   E0 + ZPE   "+"\n"
    SUBTABLE += "    -----------|-------------|------------|------------|------------|---------------------------"+"\n"

    DUo,DHo,DSo,DGo  = 0.,0.,0.,0.
    for nu_i,molecule in zip(NUS,MOLECULES):

        output_frq = files_of_interest(molecule,key[0],key[1])[3]
        if not os.path.exists(output_frq):
           allok = False
           break
        # calculate thermodynamics
        Uo, Ho, So, Go, line = compute_thermodynamics(T_ref,molecule,key,THERMODATA)
        SUBTABLE += rf"      {molecule:8s} | " + line + "\n"

        # reaction magnitudes
        DUo += nu_i * Uo
        DHo += nu_i * Ho
        DSo += nu_i * So
        DGo += nu_i * Go
    SUBTABLE += "    --------------------------------------------------------------------------------------------"+"\n"
    if not allok: continue
    print(rf"    ==> {level:s} <==")
    print("")
    print(SUBTABLE)
    # Equilibrium constant
    Kp = np.exp(-DGo/k_B/T_ref)
    # Print info
    if Kp > 1E4: ff = "9.2E"
    else       : ff = "9.3f"
    TABLE += rf"  {level:12s}      | {DUo*NA/1000:12.2f} | {DHo*NA/1000:12.2f} | {DSo*NA:12.2f} | {DGo*NA/1000:12.2f} | {Kp:{ff:s}} " + "\n"
    # save reaction magnitudes
    REFDATA[key] = T_ref,DUo*NA,DHo*NA,DSo*NA,DGo*NA
TABLE += " ----------------------------------------------------------------------------------------------" + "\n"
TABLE += "\n"
TABLE += "     units of DELTA_r{U}^o ==> kJ/mol"   + "\n"
TABLE += "     units of DELTA_r{H}^o ==> kJ/mol"   + "\n"
TABLE += "     units of DELTA_r{S}^o ==>  J/mol/K" + "\n"
TABLE += "     units of DELTA_r{G}^o ==> kJ/mol"   + "\n"

print("")
print(TABLE)
print("")

#####

The experimental value for $\Delta_{r}G^\circ$(298.15 K) is 4.74 kJ/mol, and O3LYP-D4/6-31G* provides the closest agreement among the tested levels of theory. Therefore, we will use this functional for the final step (a.3).

#### _(a.3)  $\Delta_{r}C_{p}^\circ$ and the vibrational degrees of freedom_


#####

The calculations in step (a.2) can be performed at temperatures other than 298.15 K. Let us check how the O3LYP-D4/6-31G* level predicts $\Delta_{r}G^\circ$ over the 200-400 K temperature range.

In [ ]:
#@title <small><small> { display-mode: "form" }

#------------------------------------------------------------------------------#
key = (FUNCTIONALS[1],BASIS)
T   = np.linspace(200,400,num=51)
#------------------------------------------------------------------------------#

# Calculate D_{r}G^o at different temperatures using Statistical Thermodynamics
print(rf"Computing Delta_{{r}}G^o from {T[0]:.2f} to {T[-1]:.2f} K using Statistical Mechanics...")
print("")
print(rf" - Functional: {key[0].upper():s}")
print(rf" - Basis set : {key[1].upper():s}")
print("")
DGo_T = []
for Ti in list(T):
    DGo_i = [nu_i * compute_thermodynamics(Ti,molecule,key,THERMODATA)[3] for nu_i,molecule in zip(NUS,MOLECULES)]
    DGo_i = float(sum(DGo_i)*NA)
    DGo_T.append(DGo_i)
DGo_T = np.array(DGo_T)

# Data at reference temperature
T_ref,DUo_ref,DHo_ref,DSo_ref,DGo_ref = REFDATA[key]

# Plot data
plot_DG_T(T,DGo_T,None,None,T_ref,DGo_ref,key)

#####

As you may recall from Notebook 1, the temperature dependence of the standard reaction Gibbs free energy can be written as:

$$
\Delta_{r} G^{\circ}(T) = \Delta_{r} H^{\circ}(T_{\rm ref})-T \cdot \Delta_{r} S^{\circ}(T_{\rm ref})+\Delta_{r} C_p^\circ \cdot \left[ T-T_{\rm ref}+T \cdot \ln \left(\frac{T_{\rm ref}}{T}\right)\right]
\tag{1}
$$

under the assumption that $\Delta_{r}C_{p}^\circ$ is constant. Note that we already calculated all quantities at the reference temperature ($T_{\rm ref}$ = 298.15 K), except for $\Delta_{r}C_{p}^\circ$ itself.

Next, we will explore how $\Delta_{r}C_{p}^\circ$ affects the temperature dependence of $\Delta_{r} G^{\circ}(T)$. Run the following cell as many times as you want, insert a value of $\Delta_{r}C_{p}^\circ$, and examine how well Eq. (1) reproduces the numerical data obtained in the previous calculation.

In [ ]:
#@title <small><small> { display-mode: "form" }

#------------------------------------------------------------------------------#
key = (FUNCTIONALS[1],BASIS)
T   = np.linspace(200,400,num=51)
# Data at reference temperature
T_ref,DUo_ref,DHo_ref,DSo_ref,DGo_ref = REFDATA[key]
#------------------------------------------------------------------------------#

# Values at optimized results (and other values)
print(" We can see how data fit according to the value of Delta_{r}C_{p}^o = constant * R")
DCP = float(input("   - insert value for the constant: ").strip())
DGo_model = get_DGo(T,T_ref,DHo_ref,DSo_ref,DCP*R)
RMS       = calculate_rms(DGo_T,DGo_model)/1000
print("   * Delta_{r}C_{p}^o = %5.2f * R ==> RMS = %.3f kJ/mol"%(DCP,RMS))
print("")
plot_DG_T(T,DGo_T,DGo_model,DCP*R,T_ref,DGo_ref,key)
print()

#####

The optimal value of $\Delta_{r} C_p^\circ$ can be obtained directly by fitting the computed $\Delta_{r} G^\circ(T)$ values (calculated using Statistical Thermodynamics) to Eq. (1). In other words, instead of choosing  $\Delta_{r} C_p^\circ$ manually, one could determine the value that minimizes the deviation between the analytical expression, Eq. (1), and the data, yielding the best agreement across the temperature range.

To do this, run the following cell.

In [ ]:
#@title <small><small> { display-mode: "form" }

#------------------------------------------------------------------------------#
key = (FUNCTIONALS[1],BASIS)
T   = np.linspace(200,400,num=51)
# Data at reference temperature
T_ref,DUo_ref,DHo_ref,DSo_ref,DGo_ref = REFDATA[key]
#------------------------------------------------------------------------------#

# minimize
print("Obtaining best value for Delta_{r}C_{p}^o by least squares...")
print("")
res       = minimize_scalar(sum_squared_errors,bounds=(-5,5),method='bounded',args=(T,DGo_T,T_ref,DHo_ref,DSo_ref))
DCP_best  = res.x

# Values at optimized results (and other values)
DGo_model = get_DGo(T,T_ref,DHo_ref,DSo_ref,DCP_best*R)
RMS       = calculate_rms(DGo_T,DGo_model)/1000

print("   * best fit --> Delta_{r}C_{p}^o = %5.2f * R   [RMS = %.3f kJ/mol]"%(DCP_best,RMS))
print("")

# Plotting
plot_DG_T(T,DGo_T,DGo_model,DCP_best*R,T_ref,DGo_ref,key)
print()

#####
The optimal value of $\Delta_{r} C_p^\circ$ can also be estimated from statistical mechanics. In this approach, the molar heat capacity at constant pressure of a given molecule is written as:

$$
C_{p,m} = R + \frac{3 + n^{R^\ast} + 2 \cdot n^{V^\ast}}{2} \cdot R
\tag{2}
$$

where $n^{R^\ast}$ and $n^{V^\ast}$ are the effective rotational and vibrational degrees of freedom (dofs), respectively. The term 3 accounts for the traslational dofs. For linear molecules, $n^{R^\ast}=2$; otherwise, $n^{R^\ast}=3$.

The vibrational contribution, $n^{V^\ast}$, is obtained by summing the contributions from each normal mode (frequency $\nu_j$):

$$
n^{V^\ast} = \sum_j n^{V^\ast}_j
\tag{3}
$$

Each mode contributes:

$$
n^{V^\ast}_j = \left( \frac{\theta_j^V}{T} \cdot \frac{\exp(-\theta_j^V/2T)}{1-\exp(-\theta_j^V/T)} \right)^2
\tag{4}
$$

where $\theta_j^V = h \nu_j / k_B$, and $h$ and $k_B$ are the Planck and Boltzmann constants, respectively.

To explore how $n^{V^\ast}_j$ depends on the vibrational frequency, $\nu_j$, run the following cell.

In [ ]:
#@title <small><small> { display-mode: "form" }

#------------------------------------------------------------------------------#
key = (FUNCTIONALS[1],BASIS)
T   = np.linspace(200,400,num=51)
# Data at reference temperature
T_ref,DUo_ref,DHo_ref,DSo_ref,DGo_ref = REFDATA[key]
#------------------------------------------------------------------------------#

# Find limits for the frequencies
freqmin = +float("inf")
freqmax = -float("inf")
for molecule in THERMODATA.keys():
    freqs = THERMODATA[molecule][key]["freqs"]
    freqmin = min(freqmin,min(freqs))
    freqmax = max(freqmax,max(freqs))
freqmin_cm = np.floor(freqmin/100)
freqmax_cm = np.floor(freqmax/100)

# Enable Colab’s custom widget manager, allowing interactive ipywidgets to function correctly
output.enable_custom_widget_manager()

# -------- Sliders --------
args        = dict(layout=w.Layout(width='600px'),style={'description_width': '150px'},continuous_update=True,readout_format='.2f')
freq_slider = w.FloatSlider(value=freqmin_cm,min=freqmin_cm,max=freqmax_cm,step=0.01,description=r'freq [1/cm]', **args)
ui       = w.VBox([freq_slider])

# -------- download button --------
btn      = w.Button(description='Download current figure', icon='download', button_style='primary',layout=w.Layout(width='200px', height='30px'))
btn.on_click(lambda b: _on_download_clicked(b,"vib_dof.svg"))

# -------- slider ---> function --------
# notice that P is converted from bar to Pa with *1E5
out = w.interactive_output(lambda freq: plot_average(200,400,freqmin_cm,freqmax_cm,freq), {'freq': freq_slider})
display(w.VBox([ui, btn]), out)



#####

Run the following cell to compute $C_{p,m}$ for each molecule involved in the reaction, as well as $\Delta_{r} C_p^\circ$. You will see that the value obtained is the same as the optimal value ($-0.39 \cdot R$).

In [ ]:
#@title <small><small> { display-mode: "form" }

#------------------------------------------------------------------------------#
key = (FUNCTIONALS[1],BASIS)
T   = np.linspace(200,400,num=51)
# Data at reference temperature
T_ref,DUo_ref,DHo_ref,DSo_ref,DGo_ref = REFDATA[key]
#------------------------------------------------------------------------------#

print("-------------------------------------------")
print(" Molecule | rot dof  | vib dof  |  C_{p,m} ")
print("-------------------------------------------")
DCP = 0.0
for i,molecule in enumerate(MOLECULES):
    # check if data is available
    if molecule not in THERMODATA: continue
    if key      not in THERMODATA[molecule]: continue
    # rotational dofs
    if THERMODATA[molecule][key]["islinear"]: rdof = 2
    else                                    : rdof = 3
    # calculate vdof and Cp
    freqs_cm = [ii/100 for ii in THERMODATA[molecule][key]["freqs"]]
    vdof     = sum([vib_contri_avera(freq_cm,T[0],T[-1]) for freq_cm in freqs_cm])
    Cp       = R + (3+rdof+2*vdof)/2*R
    DCP     += Cp*NUS[i]
    print(rf" {molecule:7s}  |  {rdof:6d}  |  {vdof:6.3f}  | {Cp/R:6.3f}*R  ")
print("-------------------------------------------")
print(rf"             ==> Delta_r{{C_p}}^o = {DCP/R:6.2f}*R")
print("")

### **(b) Study your own reaction**




#### 1st step: _define your reaction of interest_

In [ ]:
#@title <small><small> { display-mode: "form" }

TEXT1 = '''
===================
Firstly, introduce your reaction, using the following format:

      A + 2 B -> 3 C + 2 D

Make sure to include blank spaces between the stoichiometric coefficients and the chemical species.
For example, for the reaction we studied above, you should enter:

     N2O4 -> 2 NO2
===================

'''

TEXT2 = '''
~~~~~~~~~~~~~~~~~~~
Information:")

 * reactants (%i): %s
 * products  (%i): %s

 * equation for the reaction: %s
~~~~~~~~~~~~~~~~~~~

'''

# ---- Functions for getting the user reaction ----


def ask_for_reaction(max_nerr=3):

    nus0,molecules0 = None,None

    print(TEXT1)
    answer,nerr = "n",0
    while True:
        if nerr == max_nerr:
            print("-------------------")
            print("Too many errors... aborting...")
            print("-------------------")
            return nus0,molecules0
        try:
            reaction = input(" * insert reaction: ")
            print("")
            nus,molecules = string_to_reaction(reaction)
            if nus is None: raise Exception

            #---- Print info to make sure all is correct ----
            string = reaction_to_string(nus,molecules)
            nR = len([nu for nu in nus if nu<0])
            nP = len([nu for nu in nus if nu>0])
            sR = ", ".join([molecule for nu,molecule in zip(nus,molecules) if nu < 0])
            sP = ", ".join([molecule for nu,molecule in zip(nus,molecules) if nu > 0])
            print(TEXT2%(nR,sR,nP,sP,string))
            answer = input("Is the reaction correct? (Type 'yes' or 'no'): ")
            answer = answer.strip().lower()[0]
            if answer == "y":
               print("\nGreat! Let us continue!\n\n")
               return nus,molecules
            else:
               print("Ok, so let us try it again! :)\n\n")
               nerr += 1
        except:
            nerr += 1
            print("-------------------")
            print("There was some kind of problem... Let us try again!")

def ask_for_charge_and_unpaired(molecules):
    charges,unpaireds = {},{}
    print("Insert the total charge (a.u.) and number of unpaired electrons for each molecule (integers only):")
    print("")
    for molecule in molecules:
        print(rf"* {molecule:s}")
        charge   = input("  total molecular charge [au]  = ").strip()
        unpaired = input("  number of unpaired electrons = ").strip()
        charges[molecule]   = int(charge)
        unpaireds[molecule] = int(unpaired)
        print("")
    return charges,unpaireds
# ------------------------------------------------

# ---- Ask for reaction ----
keys = []
nus , molecules = ask_for_reaction(max_nerr=3)
if nus is not None:
   charges,unpaireds  = ask_for_charge_and_unpaired(molecules)


#### 2nd step: _retrieve geometries_

Now that the reaction is defined, we need to generate the initial guess geometries. Run the next cell to produce them.

In [ ]:
#@title <small><small> { display-mode: "form" }

#------------------------------------------------------
molecules_id  = {k:None for k in molecules}
thermodata    = {k:{}   for k in molecules}
#------------------------------------------------------

print("Getting structures. Insert one of the following:")
print("   - its SMILES code        [e.g. CCO]")
print("   - its PubChem CID number [e.g. 702]")
print("")

# Ask for CID or SMILES
for molecule in molecules:
    molecules_id[molecule] = input(rf"   * {molecule:s}: ").strip()
print("")

# Get structures
for molecule in molecules:
    print(rf"   * Obtaining guess geometry for {molecule:s}")
    mol_id    = molecules_id[molecule]
    xyz_guess = files_of_interest(molecule)[0]
    for getguess in [pubchem_cid,rdkit_smiles2geom]:
        symbols,coords,smiles = getguess(mol_id)
        if symbols is not None: break
    if symbols is None: continue
    # write file
    data_2_xyz(symbols,coords,xyz_guess,smiles)
    # visualize
    view = create_visualization_xyz(xyz_guess)
    view.show()
    print("")


#### 3rd step: _select DFT method and basis set_

Next, choose the DFT functional and basis set. Make sure that both are supported by PySCF.

Run the following cell to apply your selections and perform the geometry optimization and frequency calculation.

In [ ]:
#@title <small><small> { display-mode: "form" }

# Ask for functional and basis
functional = input("  insert DFT functional: ").strip()
basis      = input("  insert basis set     : ").strip()
print("")

# Print information
key = (functional,basis)
print(rf" - Functional: {key[0].upper():s}")
print(rf" - Basis set : {key[1].upper():s}")
print("")

# Carry on calculations
for molecule in molecules:
    print(rf" * Molecule: {molecule:s}")
    thermodata[molecule][key] = optimize_and_freqs(molecule,unpaireds[molecule],charges[molecule],key[0],key[1])
    pyscf_printdata(thermodata[molecule][key])
    pyscf_download(molecule,key[0],key[1],[1])
    print("")

    # visualization
    xyz_opt = files_of_interest(molecule,key[0],key[1])[1]
    view    = create_visualization_xyz(xyz_opt)
    view.show()
    print("")

# save successful key
if key not in keys: keys.append(key)


#### 4th step: _correct symmetry numbers_
Execute the next cell to correct the rotational symmetry numbers (if needed).

In [ ]:
#@title <small><small> { display-mode: "form" }

print("Insert the rotational symmetry number for each molecule:")
print("")
for molecule in molecules:
    sigma = int(input(rf"   * sigma({molecule:s}) = ").strip())
    print("")

    sformat = max([len(key_i[0]+key_i[1]) for key_i in keys])+1
    for key_i in keys:
        try:
          sigma_old = thermodata[molecule][key_i]["rotsigma"]
          skey = rf"{key_i[0].upper():s}/{key_i[1].upper():s}"
          print(rf"     {skey:{sformat:d}s}: {sigma_old:d} --> {sigma:d}")
          thermodata[molecule][key]["rotsigma"] = sigma
        except:
          print(rf"     data not found for {functional:s}...")
    print("")

#### 5th step: _calculate partition functions and thermodynamic functions_

Execute the next cell to select a temperature and calculate the partition functions as well as the thermodynamic functions associated to your reaction with the selected method.

In [ ]:
#@title <small><small> { display-mode: "form" }

# Ask for reference temperature
T_ref = float(input("Insert reference temperature (K): ").strip())
print("")

STRINGS = {}
STRINGS["thermo"] = ""
refdata = {}
for key_i in thermodata[molecules[0]].keys():
    # Print level of calculation
    level = rf"{key_i[0].upper():8s}/{key_i[1].upper():s}"
    STRINGS[key_i] = rf"==> {level:s} <==" + "\n\n"

    # Calculations
    allok    = True
    STRPFNS  = ""
    DUo_ref,DHo_ref,DSo_ref,DGo_ref  = 0.,0.,0.,0.
    for nu_i,molecule in zip(nus,molecules):

        output_frq = files_of_interest(molecule,key_i[0],key_i[1])[3]
        if not os.path.exists(output_frq):
            allok = False
            print("  ERROR! Something went wrong with '{molecule:s}'...")
            break
        # calculate thermodynamics
        Uo_ref, Ho_ref, So_ref, Go_ref, line = compute_thermodynamics(T_ref,molecule,key_i,thermodata)
        STRPFNS += rf"      {molecule:8s} | " + line + "\n"

        # reaction magnitudes
        DUo_ref += nu_i * Uo_ref
        DHo_ref += nu_i * Ho_ref
        DSo_ref += nu_i * So_ref
        DGo_ref += nu_i * Go_ref

    # Thermodynamic magnitudes per mol
    DUo_ref = DUo_ref*NA
    DHo_ref = DHo_ref*NA
    DSo_ref = DSo_ref*NA
    DGo_ref = DGo_ref*NA
    refdata[key_i] = T_ref, DUo_ref, DHo_ref, DSo_ref, DGo_ref

    if allok:
        # Equilibrium constant
        Kp_ref = np.exp(-DGo_ref/R/T_ref)

        # Print partition functions
        STRINGS[key_i] += "    Partition functions and zero-point–corrected total energies (E0 + ZPE); energies in hartree."    +"\n"
        STRINGS[key_i] += "\n"
        STRINGS[key_i] += "    --------------------------------------------------------------------------------------------"+"\n"
        STRINGS[key_i] += "      molecule |   pfn tr    |  pfn rot   |  pfn vib   |  pfn ele   |  pfn tot   |   E0 + ZPE   "+"\n"
        STRINGS[key_i] += "    -----------|-------------|------------|------------|------------|---------------------------"+"\n"
        STRINGS[key_i] += STRPFNS
        STRINGS[key_i] += "    --------------------------------------------------------------------------------------------"+"\n"

        # Print state functions of reaction
        if Kp_ref > 1E4: ff = "9.2E"
        else           : ff = "9.3f"
        STRINGS["thermo"] += rf"     {level:21s}| {DUo_ref/1E3:12.2f} | {DHo_ref/1E3:12.2f} | {DSo_ref:12.2f} | {DGo_ref/1E3:12.2f} | {Kp_ref:{ff:s}} " + "\n"

# print table with thermodynamics
TABLE  = "==> SUMMARY TABLE OF THERMODYNAMIC MAGNITUDES OF REACTION <=="+"\n\n"
TABLE += "    ----------------------------------------------------------------------------------------------" + "\n"
TABLE += "     FUNCTIONAL/BASIS_SET | DELTA_r{U}^o | DELTA_r{H}^o | DELTA_r{S}^o | DELTA_r{G}^o |   K_p^o   " + "\n"
TABLE += "    ----------------------|--------------|--------------|--------------|--------------|-----------" + "\n"
TABLE += STRINGS["thermo"]
TABLE += "    ----------------------------------------------------------------------------------------------" + "\n"
TABLE += "\n"
TABLE += "     units of DELTA_r{U}^o ==> kJ/mol"   + "\n"
TABLE += "     units of DELTA_r{H}^o ==> kJ/mol"   + "\n"
TABLE += "     units of DELTA_r{S}^o ==>  J/mol/K" + "\n"
TABLE += "     units of DELTA_r{G}^o ==> kJ/mol"   + "\n"
print(TABLE)

for key_i in thermodata[molecules[0]].keys():
    # print individual data for this level
    print(STRINGS[key_i])


#### 6th step: _$\Delta_{r}C_{p}^\circ$ and vibrational degrees of freedom_

In [ ]:
#@title <small><small> { display-mode: "form" }

# Temperature dependence
Tmin  = max(T_ref-150,0)
Tmax  = T_ref+150
T     = np.linspace(Tmin,Tmax,10)
for key_i in thermodata[molecules[0]].keys():
    # Print level of calculation
    level = rf"{key_i[0].upper():8s}/{key_i[1].upper():s}"
    print(rf"==> {level:s} <==" + "\n")
    # reference data
    T_ref, DUo_ref, DHo_ref, DSo_ref, DGo_ref = refdata[key_i]
    # Calculate D_{r}G^o at different temperatures using Statistical Thermodynamics
    print(rf"    Calculating Delta_r{{G}}^o between {Tmin:.2f} and {Tmax:.2f} K using Statistical Thermodynamics...")
    DGo_T = []
    for Ti in list(T):
        DGo_i = [nu_i * compute_thermodynamics(Ti,molecule,key_i,thermodata)[3] for nu_i,molecule in zip(nus,molecules)]
        DGo_T.append(float(sum(DGo_i)*NA))
    DGo_T = np.array(DGo_T)
    print("")

    # minimize
    print(rf"     - fitting to Eq. (1) using values at T_ref = {T_ref:.2f} K...")
    res       = minimize(sum_squared_errors,np.array([0.0]),args=(T,DGo_T,T_ref,DHo_ref,DSo_ref))
    dof_best  = res.x[0]

    # Values at optimized results (and other values)
    DGo_model = get_DGo(T,T_ref,DHo_ref,DSo_ref,dof_best*R)
    RMS       = calculate_rms(DGo_T,DGo_model)/1000

    print(rf"     - best fit obtained for Delta_{{r}}C_{{p}}^o = {dof_best:.2f} * R   [RMS = {RMS:.3f} kJ/mol]")
    print("")

    # Plotting
    plot_DG_T(T,DGo_T,DGo_model,dof_best*R,T_ref,DGo_ref,key_i)
    print()

    print("    -------------------------------------------")
    print("     Molecule | rot dof  | vib dof  |  C_{p,m} ")
    print("    -------------------------------------------")
    DCP = 0.0
    for i,molecule in enumerate(molecules):
        # check if data is available
        if molecule not in thermodata: continue
        if key_i    not in thermodata[molecule]: continue
        # rotational dofs
        if thermodata[molecule][key_i]["islinear"]: rdof = 2
        else                                      : rdof = 3
        # calculate vdof and Cp
        freqs_cm = [ii/100 for ii in thermodata[molecule][key_i]["freqs"]]
        vdof     = sum([vib_contri_avera(freq_cm,Tmin,Tmax) for freq_cm in freqs_cm])
        Cp       = R + (3+rdof+2*vdof)/2*R
        DCP     += Cp*nus[i]
        print(rf"      {molecule:7s}  |  {rdof:6d}  |  {vdof:6.3f}  | {Cp/R:6.3f}*R  ")
    print("     -------------------------------------------")
    print(rf"                  ==> Delta_r{{C_p}}^o = {DCP/R:6.2f}*R")
    print("")


#### 7th step: _repeat 3rd to 6th steps_

Repeat steps 3 to 6 to test additional methods.